In [ ]:
Install Packages and mount google drive

In [ ]:
!pip install lightgbm scikit-learn matplotlib seaborn -q

from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
Load data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/chicago-crime-ml/data/model1_classification.csv")
print(f"Loaded {len(df):,} rows")
print(f"\nCrime type distribution:")
print(df["primary_type"].value_counts())

In [ ]:
Prepare the features and labels 

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df["label"] = le.fit_transform(df["primary_type"])
num_classes = df["label"].nunique()
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(le.classes_)}")

feature_cols = [
    "year", "month", "day_of_week", "hour_of_day",
    "district", "ward", "community_area",
    "arrest_int", "domestic_int",
    "is_rush_hour", "is_weekend", "season",
    "location_encoded",
    "description_encoded",
    "district_hour"
]

X = df[feature_cols]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
Handle class imbalances 

In [ ]:
class_counts = y_train.value_counts().sort_index()
class_weights = len(y_train) / (num_classes * class_counts)
sample_weights = y_train.map(class_weights).values

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {le.classes_[i]}: {w:.3f}")

In [ ]:
Train lightGBM

In [ ]:
import lightgbm as lgb

N_ESTIMATORS  = 1500
LEARNING_RATE = 0.05
MAX_DEPTH     = 8
NUM_LEAVES    = 63
EARLY_STOP    = 50
RANDOM_STATE  = 42

train_data = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
val_data   = lgb.Dataset(X_test,  label=y_test,  reference=train_data)

params = {
    "objective":         "multiclass",
    "num_class":         num_classes,
    "metric":            "multi_logloss",
    "learning_rate":     LEARNING_RATE,
    "max_depth":         MAX_DEPTH,
    "num_leaves":        NUM_LEAVES,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "verbose":           -1,
    "random_state":      RANDOM_STATE,
}

print("Training LightGBM model...")
callbacks = [
    lgb.early_stopping(EARLY_STOP, verbose=True),
    lgb.log_evaluation(period=50)
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=N_ESTIMATORS,
    valid_sets=[val_data],
    callbacks=callbacks
)

print(f"\nBest iteration: {model.best_iteration}")

In [ ]:
Print the classification report 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
Feature importance plot

In [ ]:
import matplotlib.pyplot as plt

importance_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.title("LightGBM Feature Importance (Gain)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/chicago-crime-ml/outputs/lgbm_feature_importance.png", dpi=150)
plt.show()
print("Saved feature importance plot to Google Drive")

In [ ]:
Confusion matrix 

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_pct,
    annot=True, fmt=".2f",
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cmap="Blues"
)
plt.title("Confusion Matrix (row-normalized)")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/chicago-crime-ml/outputs/lgbm_confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion matrix to Google Drive")

In [ ]:
Save the model 

In [ ]:
model.save_model("/content/drive/MyDrive/chicago-crime-ml/outputs/lgbm_crime_classifier.txt")
print("Saved model to Google Drive")
print("\nStage 3a complete!")